# 📘 Company Fundamentals — SEC EDGAR

Pull a company's latest **real** financials straight from SEC filings (via `edgartools`)
— revenue, margins, profitability, and a multi-year trend. Education, **not investment advice**.


## 1. Choose a company

Set the ticker, then Run All. Data comes from the company's latest annual filing.


In [ ]:
import os
import pandas as pd
from edgar import Company, set_identity

# SEC requires a contact in the User-Agent; the app forwards EDGAR_IDENTITY if set.
set_identity(os.environ.get('EDGAR_IDENTITY', 'OpenTerminal research@openterminal.app'))

TICKER = 'AAPL'   # ← change me
fin = Company(TICKER).get_financials()
print(f'Loaded financials for {TICKER}')

## 2. Headline fundamentals


In [ ]:
rev = fin.get_revenue()
oi  = fin.get_operating_income()
ni  = fin.get_net_income()
sh  = fin.get_shares_outstanding_diluted()
eq  = fin.get_stockholders_equity()

summary = pd.DataFrame({
    'Metric': ['Revenue','Operating income','Net income','Operating margin',
               'Net margin','EPS (diluted)','Return on equity'],
    'Value':  [f'${rev/1e9:.1f}B', f'${oi/1e9:.1f}B', f'${ni/1e9:.1f}B',
               f'{oi/rev:.1%}', f'{ni/rev:.1%}', f'${ni/sh:.2f}', f'{ni/eq:.1%}'],
})
summary

## 3. Multi-year trend

Revenue and net income across the periods in the latest income statement.


In [ ]:
df = fin.income_statement().to_dataframe()
periods = [c for c in df.columns if '(' in c]

def line(*labels):
    for lbl in labels:
        m = df[df['label'].str.contains(lbl, case=False, na=False)]
        if len(m):
            return m.iloc[0][periods].astype(float)
    return None

rows = {}
rev_line = line('Revenue', 'Net sales', 'Total revenue')
ni_line  = line('Net income')
if rev_line is not None: rows['Revenue ($B)'] = (rev_line/1e9).round(1)
if ni_line  is not None: rows['Net income ($B)'] = (ni_line/1e9).round(1)
trend = pd.DataFrame(rows).T
trend.columns = [p.split(' ')[0] for p in periods]
trend

---
*Source: SEC EDGAR via edgartools. Figures are as-reported in the latest annual filing.
This notebook is educational analysis, not investment advice.*
